# Pipeline A — Lexicon counts and Fisher's exact test

A thin wrapper over `adviceaudit` for exploring Pipeline A interactively.
The canonical way to run this analysis is `make lexicon`; this notebook calls
the same library functions so results are identical.

Every identity in `config/analysis.yaml` is compared against every other,
pairwise, within each (model, prompt_number) cell.

Run from the repository root.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from adviceaudit import fishers, lexicon_counts
from adviceaudit.io_utils import load_config

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

INPUT = "../data/example/responses_example.csv"   # <-- change to your data
ANALYSIS_CONFIG = "../config/analysis.yaml"
LEXICON_CONFIG = "../config/lexicon.yaml"

## 1. Count keyword categories

In [ ]:
counts = lexicon_counts.run(
    input_path=INPUT,
    output_path="../results/notebook/counts.csv",
    analysis_config_path=ANALYSIS_CONFIG,
    lexicon_config_path=LEXICON_CONFIG,
)
print(counts.shape)
counts.head()

### How often does each category appear at all?

This is the quantity Fisher's exact test operates on: presence, not frequency.

In [ ]:
lexicon = load_config(LEXICON_CONFIG)["categories"]
count_cols = [f"count_{c}" for c in lexicon]

presence = (counts[count_cols] > 0).mean().mul(100).round(1).sort_values(ascending=False)
presence.rename("% of responses containing >=1 keyword").to_frame()

Note the caveat in `docs/lexicon_notes.md`: category coverage is very uneven
(656 terms for Religious & Spiritual, 2 for Forced Marriage), so these rates
are **not** comparable across categories. They are comparable across identities
*within* a category, which is what the test below does.

## 2. Fisher's exact tests

In [ ]:
results = fishers.run(
    input_path="../results/notebook/counts.csv",
    output_path="../results/notebook/fisher_results.csv",
    analysis_config_path=ANALYSIS_CONFIG,
    lexicon_config_path=LEXICON_CONFIG,
)
print(f"{len(results)} comparisons")
print(f"{int(results['significant_adj'].fillna(False).sum())} significant after BH correction")
results.head()

### Which identity pairs were compared?

In [ ]:
pairs = results[["identity_1", "identity_2"]].drop_duplicates().reset_index(drop=True)
print(f"{len(pairs)} distinct identity pairs (every identity vs every other):")
pairs

### Significant results only

In [ ]:
significant = results[results["significant_adj"].fillna(False)]

if significant.empty:
    print("No comparisons survived correction.")
    print("On the synthetic example this is expected: the data is random.")
else:
    display(
        significant[[
            "model", "prompt", "category", "identity_1", "identity_2",
            "pct_present_1", "pct_present_2", "pct_point_diff",
            "odds_ratio_haldane", "p_value", "p_value_adj",
        ]].round(3)
    )

`odds_ratio_haldane` is the one to plot: the plain `odds_ratio` is 0 or
infinite whenever a cell of the 2x2 table is empty, which is common with
small identities.

### Where did the tests get skipped?

In [ ]:
skipped = results[results["note"].astype(str).str.contains("skipped", na=False)]
print(f"{len(skipped)} of {len(results)} comparisons skipped for small samples")
skipped[["model", "prompt", "category", "identity_1", "identity_2", "n_1", "n_2"]].head()